# STATE Re-scoring: Top-20 DE Genes

Re-scores STATE predictions using top-20 differentially expressed genes per perturbation,
matching the evaluation metric used by the 14 benchmark methods.

**Run all cells in order.** Cell 1 installs packages and auto-restarts.

In [ ]:
# Cell 1: Install and restart (only runs pip once)
import importlib
try:
    importlib.import_module('state')
    import torch, scanpy, numpy as np
    print(f"All packages ready. numpy={np.__version__}")
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_name(0)
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU: {gpu} ({mem:.1f} GB)")
    else:
        print("WARNING: No GPU! Set Runtime > Change runtime type > T4")
    print("Proceeding to next cell...")
except ImportError:
    print("First run: installing packages...")
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'arc-state', 'scanpy', 'anndata', 'scipy',
                           'huggingface_hub'])
    print("\n" + "="*50)
    print("INSTALLED. Restarting runtime now...")
    print("After restart, click Run All again.")
    print("="*50)
    import os
    os.kill(os.getpid(), 9)

In [ ]:
# Cell 2: Cleanup + check disk space
import os, shutil

# Remove old partial downloads
hf_cache = os.path.expanduser("~/.cache/huggingface")
if os.path.exists(hf_cache):
    shutil.rmtree(hf_cache)
    print("Cleared HF cache")

# Check what we already have
DATA_PATH = "ReplogleWeissman2022_K562_essential.h5ad"
PRED_PATH = "state_predictions.h5ad"
PROC_PATH = "k562_for_state.h5ad"
MODEL_DIR = "ST-HVG-Parse/zeroshot/split_0"

for f in [DATA_PATH, PRED_PATH, PROC_PATH]:
    if os.path.exists(f):
        print(f"  FOUND: {f} ({os.path.getsize(f)/1e9:.2f} GB)")
    else:
        print(f"  MISSING: {f}")

stat = os.statvfs('/')
free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
print(f"\nFree disk space: {free_gb:.1f} GB")
assert free_gb > 5, f"Only {free_gb:.1f} GB free — need at least 5 GB"

In [ ]:
# Cell 3: Download data + model (only what's missing)
import os, urllib.request

DATA_PATH = "ReplogleWeissman2022_K562_essential.h5ad"
MODEL_DIR = "ST-HVG-Parse/zeroshot/split_0"

if not os.path.exists(DATA_PATH):
    print("Downloading K562 data (1.5 GB)...")
    url = "https://zenodo.org/records/7041849/files/ReplogleWeissman2022_K562_essential.h5ad?download=1"
    urllib.request.urlretrieve(url, DATA_PATH)
    print(f"Downloaded: {os.path.getsize(DATA_PATH)/1e9:.1f} GB")
else:
    print(f"Data present: {os.path.getsize(DATA_PATH)/1e9:.1f} GB")

if not os.path.exists(os.path.join(MODEL_DIR, "config.yaml")):
    from huggingface_hub import snapshot_download
    print("\nDownloading model (zeroshot/split_0 only, ~600 MB)...")
    snapshot_download(
        "arcinstitute/ST-HVG-Parse",
        local_dir="ST-HVG-Parse",
        allow_patterns=[
            "zeroshot/split_0/config.yaml",
            "zeroshot/split_0/checkpoints/best.ckpt",
            "zeroshot/split_0/*.pkl",
            "zeroshot/split_0/*.pt",
            "zeroshot/split_0/*.torch",
            "zeroshot/split_0/*.txt",
        ],
    )
    print("Model downloaded.")
else:
    print("Model present.")

stat = os.statvfs('/')
print(f"\nFree space: {(stat.f_bavail * stat.f_frsize)/1e9:.1f} GB")

In [ ]:
# Cell 4: Preprocess data (if not already done)
import os, gc
import scanpy as sc
from scipy import sparse

DATA_PATH = "ReplogleWeissman2022_K562_essential.h5ad"
PROC_PATH = "k562_for_state.h5ad"

if not os.path.exists(PROC_PATH):
    print("Preprocessing data...")
    adata = sc.read_h5ad(DATA_PATH)
    print(f"  Raw: {adata.shape}")
    
    # Critical: rename control label for STATE
    adata.obs["gene"] = adata.obs["gene"].replace({"non-targeting": "PBS"})
    
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    
    hvg_mask = adata.var["highly_variable"]
    X_hvg = adata[:, hvg_mask].X
    if sparse.issparse(X_hvg):
        X_hvg = X_hvg.toarray()
    adata.obsm["X_hvg"] = X_hvg
    adata.write_h5ad(PROC_PATH)
    print(f"  Saved: {PROC_PATH}")
    del adata; gc.collect()
else:
    print(f"Processed data present: {PROC_PATH}")

stat = os.statvfs('/')
print(f"Free space: {(stat.f_bavail * stat.f_frsize)/1e9:.1f} GB")

In [ ]:
# Cell 5: Run STATE inference (if predictions don't exist)
import os

PROC_PATH = "k562_for_state.h5ad"
PRED_PATH = "state_predictions.h5ad"
MODEL_DIR = "ST-HVG-Parse/zeroshot/split_0"

if not os.path.exists(PRED_PATH):
    ckpt = os.path.join(MODEL_DIR, "checkpoints", "best.ckpt")
    cmd = f"python -m state tx infer --adata_path {PROC_PATH} --model_path {MODEL_DIR} --checkpoint_path {ckpt} --output_path {PRED_PATH}"
    print(f"Running: {cmd}")
    os.system(cmd)
    assert os.path.exists(PRED_PATH), "Inference failed!"
    print(f"Predictions saved: {os.path.getsize(PRED_PATH)/1e9:.2f} GB")
else:
    print(f"Predictions present: {os.path.getsize(PRED_PATH)/1e9:.2f} GB")

In [ ]:
# Cell 6: Re-score on top-20 DE genes
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import stats, sparse
import gc, time

DATA_PATH = "ReplogleWeissman2022_K562_essential.h5ad"
PRED_PATH = "state_predictions.h5ad"
PROC_PATH = "k562_for_state.h5ad"

t0 = time.time()

# Load original data (for DE gene computation in full gene space)
print("Loading original data...")
adata_obs = sc.read_h5ad(DATA_PATH)
sc.pp.normalize_total(adata_obs, target_sum=1e4)
sc.pp.log1p(adata_obs)
all_genes = list(adata_obs.var_names)
print(f"  Original: {adata_obs.shape}")

# Load predictions
print("Loading predictions...")
adata_pred = sc.read_h5ad(PRED_PATH)
print(f"  Predictions: {adata_pred.shape}")

# Load processed data for HVG info
print("Loading processed data for HVG names...")
adata_proc = sc.read_h5ad(PROC_PATH)
if "highly_variable" in adata_proc.var.columns:
    hvg_names = list(adata_proc.var_names[adata_proc.var["highly_variable"]])
else:
    hvg_names = list(adata_proc.var_names)
hvg_set = set(hvg_names)
hvg_to_idx = {g: i for i, g in enumerate(hvg_names)}
print(f"  HVGs: {len(hvg_names)}")

gene_col = "gene"

# Control means — observed (full gene space)
ctrl_obs = adata_obs.obs[gene_col].isin(["non-targeting"])
X_ctrl_obs = adata_obs[ctrl_obs].X
if sparse.issparse(X_ctrl_obs): X_ctrl_obs = X_ctrl_obs.toarray()
ctrl_mean_full = X_ctrl_obs.mean(axis=0).flatten()

# Control means — observed (HVG space)
ctrl_obs_hvg = adata_obs[ctrl_obs][:, hvg_names].X
if sparse.issparse(ctrl_obs_hvg): ctrl_obs_hvg = ctrl_obs_hvg.toarray()
ctrl_mean_hvg_obs = ctrl_obs_hvg.mean(axis=0).flatten()
del X_ctrl_obs, ctrl_obs_hvg; gc.collect()

# Control means — predicted (HVG space)
ctrl_pred = adata_pred.obs[gene_col].isin(["PBS", "non-targeting"])
if "X_hvg" in adata_pred.obsm:
    X_pred_all = np.array(adata_pred.obsm["X_hvg"])
else:
    X_pred_all = adata_pred.X
    if sparse.issparse(X_pred_all): X_pred_all = X_pred_all.toarray()
    X_pred_all = np.array(X_pred_all)
ctrl_mean_hvg_pred = X_pred_all[ctrl_pred.values].mean(axis=0).flatten()

print(f"  Observed ctrl mean shape: {ctrl_mean_full.shape}")
print(f"  Predicted ctrl HVG shape: {ctrl_mean_hvg_pred.shape}")

# Score each perturbation
perturbations = [p for p in adata_obs.obs[gene_col].unique()
                 if p not in ["non-targeting", "PBS", "control"]]
print(f"\nScoring {len(perturbations)} perturbations...")

results_de20 = []

for i, pert in enumerate(perturbations):
    obs_mask = (adata_obs.obs[gene_col] == pert).values
    # Pred may use PBS label
    pred_mask = (adata_pred.obs[gene_col] == pert).values
    
    n_obs = obs_mask.sum()
    n_pred = pred_mask.sum()
    if n_obs < 5 or n_pred < 5:
        continue

    # Observed delta — full gene space (for finding DE genes)
    X_obs = adata_obs[obs_mask].X
    if sparse.issparse(X_obs): X_obs = X_obs.toarray()
    obs_mean_full = X_obs.mean(axis=0).flatten()
    obs_delta_full = obs_mean_full - ctrl_mean_full

    # Top-20 DE genes that are also in HVG set
    abs_delta = np.abs(obs_delta_full)
    sorted_idx = np.argsort(abs_delta)[::-1]
    de20_hvg_idx = []
    de20_full_idx = []
    for idx in sorted_idx:
        if all_genes[idx] in hvg_set:
            de20_hvg_idx.append(hvg_to_idx[all_genes[idx]])
            de20_full_idx.append(idx)
        if len(de20_hvg_idx) >= 20:
            break

    if len(de20_hvg_idx) < 5:
        continue

    # Observed delta for these DE genes (from HVG-space observed data)
    obs_hvg = adata_obs[obs_mask][:, hvg_names].X
    if sparse.issparse(obs_hvg): obs_hvg = obs_hvg.toarray()
    obs_mean_hvg = obs_hvg.mean(axis=0).flatten()
    obs_delta_hvg = obs_mean_hvg - ctrl_mean_hvg_obs
    obs_delta_de20 = obs_delta_hvg[de20_hvg_idx]

    # Predicted delta for these DE genes
    pred_mean_hvg = X_pred_all[pred_mask].mean(axis=0).flatten()
    pred_delta_hvg = pred_mean_hvg - ctrl_mean_hvg_pred
    pred_delta_de20 = pred_delta_hvg[de20_hvg_idx]

    # Pearson on DE20
    if np.std(obs_delta_de20) > 0 and np.std(pred_delta_de20) > 0:
        r, p = stats.pearsonr(obs_delta_de20, pred_delta_de20)
    else:
        r, p = 0.0, 1.0

    results_de20.append({
        "gene": pert,
        "pearson_correlation": round(r, 6),
        "pearson_pvalue": round(p, 10),
        "n_perturbed_cells": int(n_obs),
        "n_de_genes_used": len(de20_hvg_idx),
        "method": "STATE_DE20",
    })

    if (i + 1) % 100 == 0:
        print(f"  ... {i+1}/{len(perturbations)} ({time.time()-t0:.0f}s)")
    
    del X_obs, obs_hvg
    if (i + 1) % 200 == 0:
        gc.collect()

df = pd.DataFrame(results_de20)
print(f"\nScored {len(df)} perturbations")
print(f"Mean Pearson r (DE20): {df['pearson_correlation'].mean():.4f}")
print(f"Median Pearson r (DE20): {df['pearson_correlation'].median():.4f}")
print(f"\nTop 10:")
for _, row in df.nlargest(10, 'pearson_correlation').iterrows():
    print(f"  {row['gene']:15s}  r = {row['pearson_correlation']:.4f}")
print(f"\nRuntime: {time.time()-t0:.0f}s")

In [ ]:
# Cell 7: Save and download
from google.colab import files

OUT = "state_k562_de20_scores.tsv"
df.to_csv(OUT, sep="\t", index=False)
print(f"Saved {OUT} ({len(df)} genes)")

# Quick histogram
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['pearson_correlation'], bins=50, edgecolor='black', alpha=0.7)
ax.axvline(df['pearson_correlation'].mean(), color='red', ls='--',
           label=f"Mean = {df['pearson_correlation'].mean():.3f}")
ax.set_xlabel('Pearson r (top-20 DE genes)')
ax.set_ylabel('Count')
ax.set_title('STATE Performance: Re-scored on Top-20 DE Genes')
ax.legend()
plt.tight_layout()
plt.savefig('state_de20_histogram.png', dpi=150)
plt.show()

print("\nDownloading results...")
files.download(OUT)
print("\nDONE. Place the downloaded TSV in progframe/results/")